# Streaming Delta Pipeline

Demonstrate end-to-end Kafka to Bronze/Silver/Gold pipeline.

# 02 - Spark Structured Streaming + Delta Lake Pipeline

Bu notebook, Chicago Crime verisinin Kafka topic'inden Spark Structured Streaming ile okunmasını ve Delta Lake Bronze/Silver/Gold katmanlarına yazılmasını gösterir.

Pipeline akışı:

CSV → Python Kafka Producer → Kafka Topic → Spark Structured Streaming → Bronze Delta → Silver Delta → Gold Delta

## 1. Docker servislerini başlatma

Önce Kafka, Zookeeper, Python Producer, Spark Master, Spark Worker ve MLflow servisleri ayağa kaldırılır.

In [ ]:
# Terminal komutu:
# docker compose up -d
# docker compose ps

## 2. Kafka topic kontrolü

Producer mesajları `chicago_crimes_raw` topic'ine göndermektedir.

In [ ]:
# Terminal komutu:
# docker exec -it chicago_kafka kafka-topics --list --bootstrap-server chicago_kafka:9092

Beklenen topic:

```text
chicago_crimes_raw

---


## 3. Kafka Producer ile veri gönderimi

Python Producer, `data/raw/chicago_crimes_sample.csv` dosyasını okuyarak Kafka'ya JSON formatında mesaj gönderir.

Her mesajda aşağıdaki alanlar bulunmaktadır:

- `ingest_ts`

- `synthetic_user_id`

- `primary_type`

- `case_number`

- `crime_id`

- `district`

- `location_description`

- `latitude`

- `longitude`

Producer mesaj gönderim hızını `PRODUCE_RATE_PER_SEC` environment değişkeni ile ayarlanabilir hale getirir.

In [ ]:
# Terminal komutu:
# docker exec -it chicago_producer python /app/app/producer.py

Beklenen producer çıktısı:

```text
[INFO] 100 messages sent to topic 'chicago_crimes_raw'
[INFO] 200 messages sent to topic 'chicago_crimes_raw'
...
[SUCCESS] Producer finished. Total messages sent: 1000

---

## 4. Kafka → Bronze Delta

Bu adımda Spark Structured Streaming Kafka'dan JSON mesajları okur, StructType şeması ile parse eder ve Bronze Delta katmanına yazar.

Kullanılan job dosyası:

```text
jobs/01_stream_kafka_to_bronze.py

Çıktı yolu

delta/bronze/chicago_crimes_raw

In [ ]:
### Code hücresi


# Terminal komutu:
# docker exec -it chicago_spark_master /opt/spark/bin/spark-submit \
#   --master 'local[*]' \
#   --conf spark.jars.ivy=/tmp/.ivy2 \
#   --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1,io.delta:delta-spark_2.12:3.2.0 \
#   /app/jobs/01_stream_kafka_to_bronze.py

Başarılı çıktı:

```text
[SUCCESS] Bronze Delta written to: /app/delta/bronze/chicago_crimes_raw

---

## 5. Bronze → Silver

Bu adımda Bronze Delta verisi temizlenir.

Yapılan işlemler:

- `crime_id` ve `primary_type` null olan satırlar düşürülür.
- `crime_id` bazlı duplicate kayıtlar silinir.
- Sayısal kolonlarda tip dönüşümü yapılır.
- Tarih kolonu timestamp formatına çevrilir.

Kullanılan job dosyası:

```text
jobs/02_bronze_to_silver.py

Çıktı yolu

delta/silver/chicago_crimes_clean

In [ ]:
### Code hücresi

# Terminal komutu:
# docker exec -it chicago_spark_master /opt/spark/bin/spark-submit \
#   --master 'local[*]' \
#   --conf spark.jars.ivy=/tmp/.ivy2 \
#   --packages io.delta:delta-spark_2.12:3.2.0 \
#   /app/jobs/02_bronze_to_silver.py

Başarılı çıktı:

```text
[SUCCESS] Silver Delta written to: /app/delta/silver/chicago_crimes_clean

---

## 6. Silver → Gold

Bu adımda EDA ve makine öğrenmesi aşamaları için kullanılabilecek özellikler üretilir.

Üretilen bazı özellikler:

- `crime_hour`
- `crime_day_of_week`
- `crime_month`
- `is_weekend`
- `is_night`
- `arrest_int`
- `domestic_int`

Kullanılan job dosyası:

```text
jobs/03_silver_to_gold.py

Çıktı yolu

delta/gold/chicago_crimes_features

In [ ]:
### Code hücresi

# Terminal komutu:
# docker exec -it chicago_spark_master /opt/spark/bin/spark-submit \
#   --master 'local[*]' \
#   --conf spark.jars.ivy=/tmp/.ivy2 \
#   --packages io.delta:delta-spark_2.12:3.2.0 \
#   /app/jobs/03_silver_to_gold.py

SyntaxError: invalid syntax (164285130.py, line 3)

Başarılı çıktı:

```text
[INFO] Gold row count: 1000
[SUCCESS] Gold Delta written to: /app/delta/gold/chicago_crimes_features

---

## 7. Delta Lake klasör kontrolü

Delta formatının oluştuğunu doğrulamak için `_delta_log` klasörleri kontrol edilir.

In [ ]:
# Terminal komutu:
# find delta -name "_delta_log" -type d

Beklenen çıktı:

```text
delta/bronze/chicago_crimes_raw/_delta_log
delta/silver/chicago_crimes_clean/_delta_log
delta/gold/chicago_crimes_features/_delta_log

---

## Sonuç

Bu notebook ile Spark Structured Streaming + Delta Lake pipeline'ı uçtan uca doğrulanmıştır.

Tamamlanan adımlar:

- Kafka'dan JSON mesajları okundu.
- StructType ile şema tanımlandı.
- Bronze Delta katmanı oluşturuldu.
- Silver katmanında null ve duplicate temizliği yapıldı.
- Gold katmanında feature üretimi yapıldı.
- Bronze/Silver/Gold katmanlarının Delta formatında yazıldığı `_delta_log` klasörleriyle doğrulandı.